In [8]:
!pip install -q google-generativeai chromadb sentence-transformers pymupdf pdfplumber python-docx

Muat pipeline dari `banaspati_final.ipynb`

In [9]:
import json as _json
import os, glob

PIPELINE_NB = "banaspati_final.ipynb"
_cands = [PIPELINE_NB, f"/content/{PIPELINE_NB}",
          f"/content/drive/MyDrive/Colab Notebooks/AI-A2/{PIPELINE_NB}"]
_nb_path = next((p for p in _cands if os.path.exists(p)), None)
if _nb_path is None:
    for _root in ["/content", os.getcwd(), "/content/drive/MyDrive"]:
        if os.path.isdir(_root):
            hits = glob.glob(os.path.join(_root, "**", PIPELINE_NB), recursive=True)
            if hits:
                _nb_path = hits[0]; break
assert _nb_path, (f"'{PIPELINE_NB}' tidak ditemukan. Upload ke sesi Colab. "
                  f"Isi dir: {os.listdir(os.getcwd())}")
print("Memuat pipeline dari:", _nb_path)

with open(_nb_path, encoding="utf-8") as f:
    _pnb = _json.load(f)

_ip = get_ipython()
_SKIP = ("demo.launch", "demo.queue")
for _i, _c in enumerate(_pnb["cells"]):
    if _c["cell_type"] != "code":
        continue
    _src = "".join(_c["source"])
    if not _src.strip():
        continue
    if any(m in _src for m in _SKIP):
        print(f"  [skip] sel {_i} (Gradio UI)"); continue
    print(f"  [run ] sel {_i}")
    _ip.run_cell(_src)

for _name in ["retrieve", "generate_content_with_rotator", "SYSTEM_PROMPT", "collection"]:
    assert _name in dir(), f"'{_name}' tidak ada. Pastikan pipeline berjalan tanpa error."
print("Pipeline siap. Dokumen di ChromaDB:", collection.count())

Memuat pipeline dari: banaspati_final.ipynb


  [run ] sel 1
  [run ] sel 3
dataset_path = database-modul5-ai
  [run ] sel 5
  [run ] sel 7
✅ Berhasil memuat chunks dari: database-modul5-ai/all_chunks_v3.pkl
Total chunks termuat: 1389
Memuat embedding model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3843.06it/s]


Embedding model siap!
Memasukkan chunks ke ChromaDB...
Progress: 100/1389
Progress: 200/1389
Progress: 300/1389
Progress: 400/1389
Progress: 500/1389
Progress: 600/1389
Progress: 700/1389
Progress: 800/1389
Progress: 900/1389
Progress: 1000/1389
Progress: 1100/1389
Progress: 1200/1389
Progress: 1300/1389
Progress: 1389/1389
Selesai! Total dokumen di ChromaDB: 1389
  [run ] sel 9
  [run ] sel 11
System prompt dikonfigurasi dengan panduan membaca tabel jadwal
  [run ] sel 13
  [skip] sel 15 (Gradio UI)
Pipeline siap. Dokumen di ChromaDB: 1389


definisi helper demo

In [16]:
import time, json, re

# --- Konfigurasi model & harga (samakan dengan pipeline) ---
GEN_MODEL  = "gemini-3.1-flash-lite"     # model penjawab
JUDGE_MODEL = "gemini-3-flash-preview"   # Gemini 3 Flash untuk LLM-as-a-Judge
PRICE_INPUT_PER_1M  = 0.25
PRICE_OUTPUT_PER_1M = 1.50

# --- Pipeline terinstrumentasi: reuse retrieve() + generator dari pipeline ---
def run_pipeline_instrumented(pertanyaan, top_k=7):
    t0 = time.time()
    t_r = time.time()
    retrieved = retrieve(pertanyaan, top_k=top_k)
    retrieval_latency = time.time() - t_r

    contexts = [d["text"] for d in retrieved]
    sources  = [f"{d['source']} (hal. {d['page']})" for d in retrieved]

    konteks = ""
    for i, doc in enumerate(retrieved):
        konteks += f"--- SUMBER {i+1}: {doc['source']} (Halaman {doc['page']}) ---\n{doc['text']}\n\n"
    konteks = konteks.strip()

    prompt = (f"{SYSTEM_PROMPT}\n\nKONTEKS:\n{konteks}\n\n"
              f"PERTANYAAN USER:\n{pertanyaan}\n\nJAWABAN:")

    t_g = time.time()
    jawaban = generate_content_with_rotator(prompt, model_name=GEN_MODEL)
    generation_latency = time.time() - t_g
    e2e_latency = time.time() - t0

    try:
        mdl = genai.GenerativeModel(GEN_MODEL)
        input_tokens = mdl.count_tokens(prompt).total_tokens
        output_tokens = mdl.count_tokens(jawaban).total_tokens
    except Exception as e:
        print(f"warning count_tokens fallback: {e}")
        input_tokens = len(prompt.split()) * 2
        output_tokens = len(jawaban.split()) * 2
    total_tokens = input_tokens + output_tokens
    throughput = (output_tokens / generation_latency) if generation_latency > 0 else 0.0
    est_cost = (input_tokens/1_000_000)*PRICE_INPUT_PER_1M + (output_tokens/1_000_000)*PRICE_OUTPUT_PER_1M

    return {"question": pertanyaan, "answer": jawaban, "contexts": contexts, "sources": sources,
            "konteks_final": konteks, "retrieval_latency": retrieval_latency,
            "generation_latency": generation_latency, "e2e_latency": e2e_latency,
            "input_tokens": input_tokens, "output_tokens": output_tokens,
            "total_tokens": total_tokens, "throughput": throughput, "est_cost_usd": est_cost}

# --- Rubrik LLM-as-a-Judge (opsional) ---
RUBRIC = (
    "Kamu adalah evaluator (LLM-as-a-Judge) yang ketat dan objektif untuk sistem RAG akademik.\n"
    "Nilai JAWABAN SISTEM terhadap JAWABAN REFERENSI dan KONTEKS pada skala 1-5:\n"
    "- correctness, faithfulness, relevance, completeness, source_support, hallucination.\n"
    "Balas HANYA JSON: "
    '{"correctness":int,"faithfulness":int,"relevance":int,"completeness":int,'
    '"source_support":int,"hallucination":int,"reason":"singkat"}'
)

def parse_judge_json(text):
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if not m:
        raise ValueError(f"Tidak ada JSON pada output judge: {text[:200]}")
    return json.loads(m.group(0))

def demo_sandbox(pertanyaan, reference=None, top_k=7, run_judge=False):
    """Pipeline RAG penuh untuk satu pertanyaan baru + audit lengkap (soal poin 15).
    Output dirender dengan Markdown sederhana agar mudah dibaca."""
    from IPython.display import display, Markdown

    r = run_pipeline_instrumented(pertanyaan, top_k=top_k)

    md = []
    md.append(f"### 1) Pertanyaan\n{pertanyaan}")

    md.append(f"### 2) Jawaban Akhir\n{r['answer']}")

    md.append(
        "### Metrik Inferensi\n"
        "| Metrik | Nilai |\n|---|---|\n"
        f"| Retrieval latency | {r['retrieval_latency']:.3f} s |\n"
        f"| Generation latency | {r['generation_latency']:.3f} s |\n"
        f"| End-to-end latency | {r['e2e_latency']:.3f} s |\n"
        f"| Token usage | input {r['input_tokens']}, output {r['output_tokens']}, total {r['total_tokens']} |\n"
        f"| Throughput | {r['throughput']:.2f} tok/s |\n"
        f"| Estimasi biaya | ${r['est_cost_usd']:.6f} |\n"
        f"| TTFT | tidak tersedia (non-streaming) |"
    )

    ctx_lines = ["### 3) & 4) Konteks Diretrieve + Sumber"]
    for i, (ctx, src_) in enumerate(zip(r["contexts"], r["sources"]), 1):
        preview = ctx[:300].replace("\n", " ").strip()
        ctx_lines.append(f"**[{i}] {src_}**\n\n> {preview}...")
    md.append("\n\n".join(ctx_lines))

    md.append("### 5) Konteks Final yang Dikirim ke Model\n```\n" + r["konteks_final"][:3500] +
              ("\n... (dipotong untuk tampilan; konteks penuh tetap dikirim ke model) ..." if len(r["konteks_final"]) > 3500 else "") +
              "\n```")

    if run_judge and reference:
        konteks_join = "\n\n".join(f"[{i+1}] {c}" for i, c in enumerate(r["contexts"]))
        judge_prompt = (f"{RUBRIC}\n\nPERTANYAAN:\n{pertanyaan}\n\n"
                        f"JAWABAN REFERENSI:\n{reference}\n\n"
                        f"KONTEKS:\n{konteks_join}\n\n"
                        f"JAWABAN SISTEM:\n{r['answer']}\n\nPENILAIAN (JSON):")
        try:
            out = generate_content_with_rotator(judge_prompt, model_name=JUDGE_MODEL)
            sc = parse_judge_json(out)
            jt = ["### 6) LLM-as-a-Judge", "| Dimensi | Skor |", "|---|---|"]
            for k in ["correctness","faithfulness","relevance","completeness","source_support","hallucination"]:
                jt.append(f"| {k} | {sc.get(k)} |")
            jt.append(f"\n*Alasan: {sc.get('reason','-')}*")
            md.append("\n".join(jt))
        except Exception as e:
            md.append(f"### 6) LLM-as-a-Judge\n(judge gagal: {str(e)[:120]})")

    display(Markdown("\n\n---\n\n".join(md)))
    return r

print("Helper demo siap. Panggil demo_sandbox(\"pertanyaan ...\") pada sel berikut.")

Helper demo siap. Panggil demo_sandbox("pertanyaan ...") pada sel berikut.


demo

In [17]:
demo_sandbox("kamu siapa?")

### 1) Pertanyaan
kamu siapa?

---

### 2) Jawaban Akhir
Halo! Saya BANASPATI, asisten cerdas Kedai Bubur Panas IT (ITS). Saya siap membantu Anda dengan informasi seputar akademik dan operasional kedai kami. Ada yang bisa saya bantu hari ini?

---

### Metrik Inferensi
| Metrik | Nilai |
|---|---|
| Retrieval latency | 0.061 s |
| Generation latency | 2.384 s |
| End-to-end latency | 2.446 s |
| Token usage | input 2253, output 46, total 2299 |
| Throughput | 19.29 tok/s |
| Estimasi biaya | $0.000632 |
| TTFT | tidak tersedia (non-streaming) |

---

### 3) & 4) Konteks Diretrieve + Sumber

**[1] Peraturan Akademik.pdf (hal. 9)**

> 63. Mata Kuliah Intl adalah Mata Kuliah Wajib dan Mata Kuliah Pilihan yang terkait dengan program studi di ITS. 64. Mata Kuliah Non-Inti adalah Mata Kuliah di ITS yang meliputi Mata Kuliah Wajib Kurikulum, Mata Kuliah Penciri ITS, Mata Kuliah Penciri Fakultas, dan Mata Kuliah Pengayaan. 65. Mata Kul...

**[2] Visi Misi Departemen.pdf (hal. 1)**

> Butuh Bantuan? Silahkan hubungi kami disini: Bantuan TENTANG KAMI Sambutan Kepala Departemen Sejarah Departemen dalam Angka Visi dan Misi Visi Departemen Menjadi pengelola program studi bidang teknologi informasi yang memiliki reputasi internasional serta berkontribusi pada keilmuan dan kemanusiaan....

**[3] Kurikulum.pdf (hal. 6)**

> Program Studi - v IDENTITAS PROGRAM STUDI 1 Nama Perguruan Tinggi (PT) INSTITUT TEKNOLOGI SEPULUH NOPEMBER 2 Fakultas Fakultas Teknologi Elektro dan Informatika Cerdas 3 Departemen Teknologi Informasi 4 Program Studi S1 Teknologi Informasi 5 Status Akreditasi Baik Sekali 6 Jumlah Mahasiswa 257 7 Jum...

**[4] Peraturan Akademik.pdf (hal. 8)**

> Dosen wall adalah dosen yang bertugas membantu mahasiswa dan memantau perkembangan studi Mahasiswa sejak awal hirigga mahasiswa dinyatakan lulus. 56. Calon Mahasiswa adalah peserta penerinaan mahasiswa barn yang telah mendaftarkan din i dan mempunyai kartu peserta. 57. Calon mahasiswa barn adalah pe...

**[5] Data Dosen.pdf (hal. 4)**

> Nama Dosen Dr. Ir. Henning Titi Ciptaningtyas, S.Kom, M.Kom. NIP NIDN 0008078402 Jabatan Dosen Email henning@its.ac.id mailto:henning@if.its.ac.id); henning.its@gmail.com mailto:henning.its@gmail.com) Riwayat Pendidikan S-1, Institut Teknologi Sepuluh Nopember, 2006 Magister, Insititut Teknologi Sep...

**[6] Data Dosen.pdf (hal. 2)**

> Nama Dosen Dr.techn. Ir. Raden Venantius Hari Ginardi, M.Sc. NIP NIDN 0018056508 Jabatan Kepala Departemen Teknologi Informasi Email hari@its.ac.id mailto:hari@its.ac.id); hari.ginardi@gmail.com mailto:hari.ginardi@gmail.com) Riwayat Pendidikan S-1, Institut Teknologi Sepuluh Nopember, 1991 Magister...

**[7] Visi Misi Departemen.pdf (hal. 2)**

> Visi Keilmuan Program Studi Sarjana Teknologi Informasi Menghasilkan sarjana yang unggul secara nasional serta mampu bersaing di tingkat internasional dan berkontribusi bagi masyarakat pada bidang ilmu teknologi informasi, khususnya dalam kompetensi Cybersecurity, System Integration dan Cloud Comput...

---

### 5) Konteks Final yang Dikirim ke Model
```
--- SUMBER 1: Peraturan Akademik.pdf (Halaman 9) ---
63. Mata Kuliah Intl adalah Mata Kuliah Wajib dan Mata Kuliah Pilihan yang terkait dengan program studi di ITS. 64. Mata Kuliah Non-Inti adalah Mata Kuliah di ITS yang meliputi Mata Kuliah Wajib Kurikulum, Mata Kuliah Penciri ITS, Mata Kuliah Penciri Fakultas, dan Mata Kuliah Pengayaan. 65. Mata Kuliah Wajib adalah Mata Kuliah yang1wajib bagi Mahasiswa Program Studi untuk memenuhi syarat kelulusan. 66. Mata Kuliah Pilihan adalah Mata Kuliah penunjang keahlian khusus Program Studi sesuai bidang minat. 67. Mata Kuliah Wajib Kurikulum yang selanjutnya disebut MKWK adalah Mata Kuliah yang terdiri dan Agama, Pancasila, Kewarganegaraan, dan Bahasa Indonesia. 68. Mata Kuliah Penciri ITS untuk Program Sarjana dan Sarjana Terapan adalah Mata Kuliah yang terdiri dan Aplikasi Teknologi dan Transformasi Digital, Kewirausahaan Berbasis Teknologi, dan Bahasa Inggris. 69. Mata Kuliah Penciri Fakultas adalah Mata Kuliah yang diselenggarakan oleh Fakultas dan bersifat opsional. 70. Mata Kuliah Pengayaan pada satu Program Studi adalah Mata Kuliah yang ditawarkan untuk Mahasiswa Program Studi lain yang bersifat memperkaya pengetahuan dan/atau pengalaman. 71. Tahap Persiapan adalah tahap pembelajaran yang dijadwalkan dalam dua semester pertama pada kurikulum Program Sarjana atau dua paket semester pertama pada kurikulum Program Sarjana Terapan. 72.

--- SUMBER 2: Visi Misi Departemen.pdf (Halaman 1) ---
Butuh Bantuan? Silahkan hubungi kami disini: Bantuan TENTANG KAMI Sambutan Kepala Departemen Sejarah Departemen dalam Angka Visi dan Misi Visi Departemen Menjadi pengelola program studi bidang teknologi informasi yang memiliki reputasi internasional serta berkontribusi pada keilmuan dan kemanusiaan. VISI DAN MISI 15/04/26, 15.36 Visi dan Misi - Departemen Teknologi Informasi 1/3

--- SUMBER 3: Kurikulum.pdf (Halaman 6) ---
Program Studi - v IDENTITAS PROGRAM STUDI 1 Nama Perguruan Tinggi (PT) INSTITUT TEKNOLOGI SEPULUH NOPEMBER 2 Fakultas Fakultas Teknologi Elektro dan Informatika Cerdas 3 Departemen Teknologi Informasi 4 Program Studi S1 Teknologi Informasi 5 Status Akreditasi Baik Sekali 6 Jumlah Mahasiswa 257 7 Jumlah Dosen 11 8 Alamat Prodi Gedung Perpustakaan ITS Lt. 6 dan Gedung Tower 2 Lt. 2 Kampus ITS Sukolilo, Surabaya, Jawa Timur 60111 9 Telp +62 31-5994251-54 / +62 81333848401 10 Web Prodi/Dep. https://www.its.ac.id/it/

--- SUMBER 4: Peraturan Akademik.pdf (Halaman 8) ---
Dosen wall adalah dosen yang bertugas membantu mahasiswa dan memantau perkembangan studi Mahasiswa sejak awal hirigga mahasiswa dinyatakan lulus. 56. Calon Mahasiswa adalah peserta penerinaan mahasiswa barn yang telah mendaftarkan din i dan mempunyai kartu peserta. 57. Calon mahasiswa barn adalah peserta sel'eksi yang dinyatakan lulus seleksi penerimaan mahasiswa barn. 58. Keinsinyuran adalah kegiatan teknik dengan menggunakan kepakaran dan keahlian berdasarkan penguasaan ilmu pengetahuan dan teknologi untuk meningkatkan nilai i tambah dan daya guna secara berkelanjutan dengan memperhatikan keselamatan, kesehatan, kemaslahatan, serta masyarakat dan kelestarian Iingkungan. 59. Arsitek adalah seseorang yang melakukan Praktik Arsitek dan telah memenuhi syarat dan ditetapkan oleh dewan untuk meaakukan Praktik Arsitek. 60. Praktik Arsitek adalah penyelenggaraan kegiatan untuk menghasilkan karya Arsitektur yang meliputi perencanaan, perancangan, pengawasan, dan/atau pengkajian untuk bangunan gedung dan lingkungannya, serta yang ter
... (dipotong untuk tampilan; konteks penuh tetap dikirim ke model) ...
```

{'question': 'kamu siapa?',
 'answer': 'Halo! Saya BANASPATI, asisten cerdas Kedai Bubur Panas IT (ITS). Saya siap membantu Anda dengan informasi seputar akademik dan operasional kedai kami. Ada yang bisa saya bantu hari ini?',
 'contexts': ['63. Mata Kuliah Intl adalah Mata Kuliah Wajib dan Mata Kuliah Pilihan yang terkait dengan program studi di ITS. 64. Mata Kuliah Non-Inti adalah Mata Kuliah di ITS yang meliputi Mata Kuliah Wajib Kurikulum, Mata Kuliah Penciri ITS, Mata Kuliah Penciri Fakultas, dan Mata Kuliah Pengayaan. 65. Mata Kuliah Wajib adalah Mata Kuliah yang1wajib bagi Mahasiswa Program Studi untuk memenuhi syarat kelulusan. 66. Mata Kuliah Pilihan adalah Mata Kuliah penunjang keahlian khusus Program Studi sesuai bidang minat. 67. Mata Kuliah Wajib Kurikulum yang selanjutnya disebut MKWK adalah Mata Kuliah yang terdiri dan Agama, Pancasila, Kewarganegaraan, dan Bahasa Indonesia. 68. Mata Kuliah Penciri ITS untuk Program Sarjana dan Sarjana Terapan adalah Mata Kuliah yang te